In [5]:
import asyncio
import aiohttp
from concurrent.futures import ThreadPoolExecutor
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from typing import List, Dict, Set
import json
import os

class CryptoDataFetcher:
    def __init__(self, api_key: str, output_file: str = None):
        self.api_key = api_key
        self.base_url = "https://financialmodelingprep.com/stable"
        
        self.COLUMN_ORDER = [
            'symbol', 'name', 'price', 'changesPercentage', 'change', 
            'dayLow', 'dayHigh', 'yearHigh', 'yearLow', 
            'marketCap', 'priceAvg50', 'priceAvg200', 'volume', 
            'avgVolume', 'open', 'previousClose', 'eps', 'pe', 
            'earningsAnnouncement', 'sharesOutstanding', 'timestamp',
            'daily_range_pct', 'ytd_gain_pct', 'distance_from_50ma_pct',
            'distance_from_200ma_pct', 'market_cap_category', 
            'volume_to_mcap_ratio', 'data_collected_at'
        ]
        
        if output_file is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            output_file = f"crypto_data_{timestamp}.csv"
        
        self.output_file = output_file
        self.output_path = f"../outputs/{self.output_file}"
        self.checkpoint_file = f"../outputs/.{output_file}.checkpoint"
        
    def fetch_crypto_list(self) -> List[Dict]:
        """Fetch the complete list of cryptocurrencies"""
        url = f"{self.base_url}/cryptocurrency-list?apikey={self.api_key}"
        
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            print(f"✓ Fetched {len(data)} cryptocurrencies")
            return data
        except Exception as e:
            print(f"✗ Error fetching crypto list: {e}")
            return []
    
    def fetch_quote(self, symbol: str) -> Dict:
        """Fetch detailed quote data for a specific symbol"""
        url = f"{self.base_url}/quote?symbol={symbol}&apikey={self.api_key}"
        
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if data and len(data) > 0:
                return data[0]
            return None
        except Exception as e:
            return None
    
    def get_already_fetched_symbols(self) -> Set[str]:
        """Get list of symbols already in the output file"""
        if not os.path.exists(self.output_path):
            return set()
        
        try:
            df = pd.read_csv(self.output_path)
            if 'symbol' in df.columns:
                fetched = set(df['symbol'].unique())
                print(f"📂 Found {len(fetched)} already-fetched symbols in existing file")
                return fetched
        except Exception as e:
            print(f"⚠️  Could not read existing file: {e}")
        
        return set()
    
    def _add_derived_metrics(self, row: Dict) -> Dict:
        """Add derived statistical features to a single row"""
        
        if 'dayHigh' in row and 'dayLow' in row and 'price' in row:
            if row['price'] and row['dayHigh'] and row['dayLow'] and row['price'] != 0:
                row['daily_range_pct'] = round((row['dayHigh'] - row['dayLow']) / row['price'] * 100, 2)
        
        if 'price' in row and 'yearLow' in row:
            if row['price'] and row['yearLow'] and row['yearLow'] != 0:
                row['ytd_gain_pct'] = round((row['price'] - row['yearLow']) / row['yearLow'] * 100, 2)
        
        if 'price' in row and 'priceAvg50' in row:
            if row['price'] and row['priceAvg50'] and row['priceAvg50'] != 0:
                row['distance_from_50ma_pct'] = round((row['price'] - row['priceAvg50']) / row['priceAvg50'] * 100, 2)
        
        if 'price' in row and 'priceAvg200' in row:
            if row['price'] and row['priceAvg200'] and row['priceAvg200'] != 0:
                row['distance_from_200ma_pct'] = round((row['price'] - row['priceAvg200']) / row['priceAvg200'] * 100, 2)
        
        if 'marketCap' in row and row['marketCap']:
            mcap = row['marketCap']
            if mcap < 1e7:
                row['market_cap_category'] = 'Micro (<10M)'
            elif mcap < 1e8:
                row['market_cap_category'] = 'Small (10M-100M)'
            elif mcap < 1e9:
                row['market_cap_category'] = 'Mid (100M-1B)'
            elif mcap < 1e10:
                row['market_cap_category'] = 'Large (1B-10B)'
            else:
                row['market_cap_category'] = 'Mega (>10B)'
        
        if 'volume' in row and 'marketCap' in row:
            if row['volume'] and row['marketCap'] and row['marketCap'] != 0:
                row['volume_to_mcap_ratio'] = round(row['volume'] / row['marketCap'] * 100, 2)
        
        row['data_collected_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        return row
    
    def write_batch_to_csv(self, rows: List[Dict], is_first: bool = False):
        """Write multiple rows to CSV at once"""
        if not rows:
            return
            
        df = pd.DataFrame(rows)
        df = df.reindex(columns=self.COLUMN_ORDER)
        
        write_header = is_first and not os.path.exists(self.output_path)
        df.to_csv(self.output_path, mode='a', header=write_header, index=False)
    
    def save_checkpoint(self, processed_symbols: Set[str]):
        """Save checkpoint of processed symbols"""
        with open(self.checkpoint_file, 'w') as f:
            json.dump(list(processed_symbols), f)
    
    def load_checkpoint(self) -> Set[str]:
        """Load checkpoint of processed symbols"""
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    return set(json.load(f))
            except:
                pass
        return set()
    
    def fetch_all_crypto_data(self, sample_size: int = None, 
                             max_workers: int = 20,
                             batch_size: int = 100) -> str:
        """
        Fetch comprehensive data for all cryptocurrencies with parallel processing
        
        Args:
            sample_size: If specified, only fetch this many cryptos (for testing)
            max_workers: Number of concurrent threads (default: 20)
            batch_size: Write to CSV every N cryptos (default: 100)
        
        Returns:
            Path to output file
        """
        print("=" * 70)
        print("CRYPTOCURRENCY DATA COLLECTION (Parallel Mode)")
        print("=" * 70)
        
        # Step 1: Get list of all cryptocurrencies
        print("\n[1/4] Fetching cryptocurrency list...")
        crypto_list = self.fetch_crypto_list()
        
        if not crypto_list:
            print("No cryptocurrencies found. Exiting.")
            return None
        
        if sample_size:
            crypto_list = crypto_list[:sample_size]
            print(f"📊 Processing sample of {sample_size} cryptocurrencies")
        
        # Step 2: Check what's already been fetched
        print(f"\n[2/4] Checking for existing data...")
        already_fetched = self.get_already_fetched_symbols()
        
        symbols_to_fetch = [c for c in crypto_list if c['symbol'] not in already_fetched]
        
        if not symbols_to_fetch:
            print("✓ All symbols already fetched! Nothing to do.")
            return self.output_path
        
        print(f"📋 Need to fetch: {len(symbols_to_fetch)} symbols")
        print(f"✓ Already have: {len(already_fetched)} symbols")
        print(f"🎯 Total: {len(crypto_list)} symbols")
        
        # Step 3: Fetch data in parallel
        print(f"\n[3/4] Fetching data with {max_workers} parallel workers...")
        print()
        
        successful = 0
        failed = 0
        batch_buffer = []
        is_first_write = not os.path.exists(self.output_path)
        
        start_time = time.time()
        
        # Process in parallel using ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all tasks
            future_to_crypto = {
                executor.submit(self.fetch_quote, crypto['symbol']): crypto 
                for crypto in symbols_to_fetch
            }
            
            # Process completed tasks
            for idx, future in enumerate(future_to_crypto, 1):
                crypto = future_to_crypto[future]
                symbol = crypto['symbol']
                
                # Progress indicator
                if idx % 50 == 0 or idx == len(symbols_to_fetch):
                    elapsed = time.time() - start_time
                    rate = idx / elapsed if elapsed > 0 else 0
                    remaining = len(symbols_to_fetch) - idx
                    eta_seconds = remaining / rate if rate > 0 else 0
                    eta_str = str(timedelta(seconds=int(eta_seconds)))
                    
                    print(f"Progress: {idx}/{len(symbols_to_fetch)} ({idx/len(symbols_to_fetch)*100:.1f}%) | "
                          f"Success: {successful} | Failed: {failed} | "
                          f"Rate: {rate:.1f}/s | ETA: {eta_str}")
                
                try:
                    quote_data = future.result()
                    
                    if quote_data:
                        combined = {**crypto, **quote_data}
                        combined = self._add_derived_metrics(combined)
                        
                        batch_buffer.append(combined)
                        successful += 1
                        already_fetched.add(symbol)
                        
                        # Write batch when buffer is full
                        if len(batch_buffer) >= batch_size:
                            self.write_batch_to_csv(batch_buffer, is_first=is_first_write)
                            is_first_write = False
                            batch_buffer = []
                            self.save_checkpoint(already_fetched)
                    else:
                        failed += 1
                        
                except Exception as e:
                    failed += 1
        
        # Write remaining buffer
        if batch_buffer:
            self.write_batch_to_csv(batch_buffer, is_first=is_first_write)
        
        self.save_checkpoint(already_fetched)
        
        # Step 4: Summary
        elapsed_total = time.time() - start_time
        print("\n" + "=" * 70)
        print("COLLECTION SUMMARY")
        print("=" * 70)
        print(f"✓ Successfully fetched: {successful} cryptocurrencies")
        print(f"✗ Failed to fetch: {failed} cryptocurrencies")
        print(f"⏱️  Total time: {str(timedelta(seconds=int(elapsed_total)))}")
        print(f"📈 Average rate: {successful/elapsed_total:.1f} cryptos/second")
        print(f"💾 Output file: {self.output_file}")
        print(f"📊 Total symbols in file: {len(already_fetched)}")
        print("=" * 70)
        
        return self.output_path


def main():
    API_KEY = "m8TZJWQFGH7G6x2nowAqKdzDfAyakr0T"
    
    fetcher = CryptoDataFetcher(
        API_KEY, 
        output_file="crypto_market.csv")

    
    # Parallel processing - MUCH FASTER!
    filepath = fetcher.fetch_all_crypto_data(
        sample_size=None,  # Set to None for all cryptos
        max_workers=20,    # Adjust based on API rate limits (10-50 recommended)
        batch_size=100     # Write every 100 cryptos
    )
    
    if filepath:
        print(f"\n✅ Data collection complete!")
        print(f"📁 File location: {filepath}")
        
        df = pd.read_csv(filepath)
        
        print("\n" + "=" * 70)
        print("DATA PREVIEW (First 5 rows)")
        print("=" * 70)
        print(df.head().to_string())
        
        print("\n" + "=" * 70)
        print("KEY STATISTICS")
        print("=" * 70)
        print(f"Total rows: {len(df):,}")
        print(f"Total columns: {len(df.columns)}")
        
        if 'marketCap' in df.columns:
            print(f"Total Market Cap: ${df['marketCap'].sum():,.0f}")
        if 'volume' in df.columns:
            print(f"Total 24h Volume: ${df['volume'].sum():,.0f}")
        if 'price' in df.columns:
            print(f"Average Price: ${df['price'].mean():,.2f}")
            print(f"Median Price: ${df['price'].median():,.2f}")
    else:
        print("\n⚠️  No data collected.")


if __name__ == "__main__":
    main()

CRYPTOCURRENCY DATA COLLECTION (Parallel Mode)

[1/4] Fetching cryptocurrency list...
✓ Fetched 4785 cryptocurrencies

[2/4] Checking for existing data...
📋 Need to fetch: 4785 symbols
✓ Already have: 0 symbols
🎯 Total: 4785 symbols

[3/4] Fetching data with 20 parallel workers...

Progress: 50/4785 (1.0%) | Success: 49 | Failed: 0 | Rate: 19.2/s | ETA: 0:04:06
Progress: 100/4785 (2.1%) | Success: 99 | Failed: 0 | Rate: 22.2/s | ETA: 0:03:30
Progress: 150/4785 (3.1%) | Success: 149 | Failed: 0 | Rate: 22.0/s | ETA: 0:03:30
Progress: 200/4785 (4.2%) | Success: 199 | Failed: 0 | Rate: 22.6/s | ETA: 0:03:22
Progress: 250/4785 (5.2%) | Success: 249 | Failed: 0 | Rate: 22.3/s | ETA: 0:03:23
Progress: 300/4785 (6.3%) | Success: 299 | Failed: 0 | Rate: 21.4/s | ETA: 0:03:29
Progress: 350/4785 (7.3%) | Success: 349 | Failed: 0 | Rate: 22.1/s | ETA: 0:03:20
Progress: 400/4785 (8.4%) | Success: 399 | Failed: 0 | Rate: 22.5/s | ETA: 0:03:14
Progress: 450/4785 (9.4%) | Success: 449 | Failed: 0 | R